# 阶段 12 验证：连续参数 + 四象限统计

配合 `verify.md` 和 `docs/05_optimization_plan.md` 的步骤 2。
三个检查，按顺序跑：

1. **PARAM 真的连续可调**（最关键）—— 性能表应该单调变化；
2. **四象限统计在跑** —— 遮住一个角，对应象限的值要掉；
3. **自适应 vs 固定** —— 出答辩要用的对比数据。

跑之前确认板上是阶段 12 的 bitstream（`VERSION` 末尾应该是 `0x0C`）。


In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT):
    if os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
        break
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib import adaptive
from lib.ai_video_regs import AiVideoRegs

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)

print("VERSION    = 0x%08X" % regs.version())
print("阶段 = %d  修订 = %d" % (regs.version_stage(), regs.version_revision()))
print("has_param  =", regs.has_param())

assert regs.version_stage() >= 12, "板上不是阶段 12 的 bitstream，先跑 build_overlay.sh 12"
print("默认 PARAM =", regs.param(), "（应为 192）")

vp.start_capture(vdma0, mode)
regs.wait_new_frame()
print("采集已启动。请让画面保持不动（后面所有数字才有可比性）。")


## 检查 1：PARAM 真的连续可调（最关键）

固定模式、扫 PARAM，看三个指标是不是**单调变化**：

* mode 0 增益：**均值**单调升；PARAM=128 时饱和率应该 ≈ 0（阶段 7 在这里是 26.7%）
* mode 1 对比拉伸：**std** 单调升；而且中间调也会变（阶段 7 对 96~160 完全不动）
* mode 2 锐化：**水平差分**单调升；PARAM=128 ≈ 阶段 7 的效果

> 场景别用纯白墙：mode 1/2 需要画面里有明暗/纹理才看得出来。


In [ ]:
def shot(mode, param=None, warm=4):
    regs.set_mode(mode)
    if param is not None:
        regs.set_param(param)
    for _ in range(warm):
        regs.wait_new_frame()
    raw = vp.capture_frame(vdma0)
    return vp.frame_to_rgb(raw).astype(np.float32)


PARAMS = (128, 160, 192, 224, 255)
print("模式  PARAM    均值     std   水平差分   饱和%")
print("-" * 54)
for m in (0, 1, 2):
    for p in PARAMS:
        f = shot(m, p)
        print("  %d     %3d   %7.1f %7.1f %8.2f %7.1f"
              % (m, p, f.mean(), f.std(),
                 np.abs(np.diff(f, axis=1)).mean(), 100.0 * (f >= 255).mean()))
    print("-" * 54)
print("判据：每一档的对应列都要单调变化；否则把这列贴给我。")


## 检查 2：四象限统计在跑

先读一次，然后**用手遮住画面左上角**（或者用黑卡纸贴住左上角），再读一次 —— 左上那个值应该明显下降，其余三个基本不动。


In [ ]:
def quads_now(tag=""):
    for _ in range(3):
        regs.wait_new_frame()
    q = regs.quads()
    print("%-10s 左上=%5d 右上=%5d 左下=%5d 右下=%5d   不均匀度=%.3f"
          % (tag, q[0], q[1], q[2], q[3], regs.spatial_unevenness()))
    return q, regs.spatial_unevenness()


q1, u1 = quads_now("before 遮之前")
input(">>> 现在用手/黑卡纸遮住画面【左上角】，然后按 Enter 继续")
q2, u2 = quads_now("after  遮之后")

print()
print("左上变化 = %+d（应该明显变负）" % (q2[0] - q1[0]))
print("另外三个变化 = %+d %+d %+d（应该接近 0）"
      % (q2[1]-q1[1], q2[2]-q1[2], q2[3]-q1[3]))
print("不均匀度 %.3f -> %.3f（遮一个角后应该变大）" % (u1, u2))


## 检查 3：固定 vs 连续自适应（答辩用的对比数据）

同一个场景下比较两种策略的画质指标。
**预期结论**：固定模式在暗光下提亮不足、在正常亮度下过曝；
自适应把增益按亮度反推，饱和率降下来、有效对比度提上去。


In [ ]:
def quality(a):
    return (float(a.mean()), float(a.std()),
            float(np.abs(np.diff(a, axis=1)).mean()),
            100.0 * float((a >= 255).mean()))


print("场景  策略      PARAM  均值    std   水平差分  饱和%")
print("-" * 62)
for name, m in (("暗光", 0), ("逆光", 1), ("常规", 2)):
    # 固定：用阶段 7 的默认强度（PARAM=192）
    a = shot(m, 192)
    print("%-4s  固定       192  %6.1f %6.1f %8.2f %6.1f" % ((name,) + quality(a)))
    # 自适应：按当前特征算 PARAM
    f = regs.mlp_features()
    # decide() returns (mode, param, reason) -- keep only param and reason
    _, p, why = adaptive.decide(f, m)
    b = shot(m, p)
    print("%-4s  自适应     %3d  %6.1f %6.1f %8.2f %6.1f   (%s)"
          % ((name, p) + quality(b) + (why,)))
    print("-" * 62)
print("把这张表贴进报告，标题写：固定强度 vs 连续自适应的画质对比")


## 收尾


In [ ]:
regs.set_mode(0)
regs.set_param(192)
vs.stop_all_channels(vdma0, vdma1)
print("[ok] 模式恢复到 0 / PARAM=192，VDMA 通道已停")
try:
    ol.free()
    print("[ok] overlay 已释放")
except Exception as e:
    print("[warn] 释放 overlay 失败：%s" % e)
print()
print("下一步：07_auto_schedule.ipynb 第 10 格的 cmd 加上 --adaptive，跑连续自适应调度")
